<a href="https://colab.research.google.com/github/Quijano89/Trial1_codekada/blob/main/Codekada_backend_connected.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import cv2
import numpy as np
import os
import csv

# -------------------------
# CONFIG
# -------------------------
video_path = "actualspring.mp4"
output_csv = "SPRING1.csv"

print("Working dir:", os.getcwd())

if not os.path.exists(video_path):
    raise FileNotFoundError("Video not found.")

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise RuntimeError("Video failed to open.")

ret, frame = cap.read()
if not ret:
    raise RuntimeError("Cannot read first frame.")

# -------------------------
# DISPLAY FIX
# -------------------------
display_scale = 0.6
cv2.namedWindow("Stable Physics Tracking", cv2.WINDOW_NORMAL)

# -------------------------
# TRACKER
# -------------------------
def create_tracker():
    if hasattr(cv2, "legacy") and hasattr(cv2.legacy, "TrackerCSRT_create"):
        return cv2.legacy.TrackerCSRT_create()
    elif hasattr(cv2, "TrackerCSRT_create"):
        return cv2.TrackerCSRT_create()
    else:
        return cv2.TrackerKCF_create()

# -------------------------
# ROI SELECTION (SCALED ONLY FOR UI)
# -------------------------
roi_scale = 0.5
frame_small = cv2.resize(frame, (0, 0), fx=roi_scale, fy=roi_scale)

bbox_small = cv2.selectROI("Select Object", frame_small, False)
cv2.destroyAllWindows()

x, y, w, h = bbox_small

# scale back to full resolution
bbox = (
    int(x / roi_scale),
    int(y / roi_scale),
    int(w / roi_scale),
    int(h / roi_scale)
)

tracker = create_tracker()
tracker.init(frame, bbox)

# -------------------------
# STATE
# -------------------------
centers = []
trail = []

lost_frames = 0
max_lost = 25

fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30

smooth_cx, smooth_cy = None, None
vx, vy = 0.0, 0.0

alpha = 0.35
beta = 0.75

# -------------------------
# MAIN LOOP
# -------------------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    success, bbox = tracker.update(frame)

    if success:
        lost_frames = 0

        x, y, w, h = map(int, bbox)

        roi = frame[y:y+h, x:x+w]
        if roi.size == 0:
            continue

        gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (5, 5), 0)

        _, mask = cv2.threshold(
            gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )

        M = cv2.moments(mask)

        if M["m00"] != 0:
            dx = M["m10"] / M["m00"]
            dy = M["m01"] / M["m00"]
        else:
            dx, dy = w / 2, h / 2

        cx = x + dx
        cy = y + dy

        # outlier rejection
        if len(centers) > 2:
            px, py = centers[-1]
            if abs(cx - px) > 150 or abs(cy - py) > 150:
                continue

        # smoothing
        if smooth_cx is None:
            smooth_cx, smooth_cy = cx, cy
            vx, vy = 0.0, 0.0
        else:
            vx = beta * vx + (1 - beta) * (cx - smooth_cx)
            vy = beta * vy + (1 - beta) * (cy - smooth_cy)

            smooth_cx += vx
            smooth_cy += vy

            smooth_cx = alpha * cx + (1 - alpha) * smooth_cx
            smooth_cy = alpha * cy + (1 - alpha) * smooth_cy

        centers.append((smooth_cx, smooth_cy))
        trail.append((int(smooth_cx), int(smooth_cy)))

        if len(trail) > 500:
            trail = trail[-500:]

        # draw bbox
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

        # draw center
        cv2.circle(
            frame,
            (int(smooth_cx), int(smooth_cy)),
            4,
            (0, 0, 255),
            -1
        )

        # trail
        start = max(1, len(trail) - 120)
        for i in range(start, len(trail)):
            cv2.line(frame, trail[i - 1], trail[i], (255, 0, 0), 2)

    else:
        lost_frames += 1

        cv2.putText(
            frame,
            f"Lost ({lost_frames}/{max_lost})",
            (50, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 0, 255),
            2
        )

        if lost_frames > max_lost:
            bbox_small = cv2.selectROI("Re-select object", frame, False)
            cv2.destroyAllWindows()

            x, y, w, h = bbox_small
            bbox = (x, y, w, h)

            tracker = create_tracker()
            tracker.init(frame, bbox)

            lost_frames = 0
            trail.clear()
            smooth_cx, smooth_cy = None, None
            vx, vy = 0.0, 0.0

    # display
    frame_display = cv2.resize(
        frame, (0, 0), fx=display_scale, fy=display_scale
    )
    cv2.imshow("Stable Physics Tracking", frame_display)

    wait_time = max(1, int(1000 / fps))
    if cv2.waitKey(wait_time) & 0xFF == ord('q'):
        break

# -------------------------
# SAVE DATA CONTRACT
# -------------------------
cap.release()
cv2.destroyAllWindows()

if len(centers) > 0:

    with open(output_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["x", "y"])
        for c in centers:
            writer.writerow([c[0], c[1]])

    np.save("centers.npy", centers)
    np.save("dt.npy", 1 / fps)

    print("\nDATA EXPORT COMPLETE")
    print("--------------------")
    print("samples:", len(centers))
    print("fps:", fps)
    print("saved csv:", output_csv)
    print("saved npy: centers.npy, dt.npy")

else:
    print("No data collected.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import csv
from scipy.signal import savgol_filter, find_peaks
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-darkgrid")

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "lines.linewidth": 2,
    "legend.frameon": True,
})

# -------------------------
# LOAD CSV
# -------------------------
t = []
y = []

with open("SPRING1.csv", "r") as f:
    reader = csv.reader(f)
    next(reader)

    for i, row in enumerate(reader):
        yy = float(row[1])
        t.append(i)
        y.append(yy)

t = np.array(t, dtype=float)
y = np.array(y, dtype=float)

# -------------------------
# TIME SCALE
# -------------------------
fps = 30
t = t / fps
dt = 1 / fps

print("\nDATA CHECK")
print("samples:", len(y))
print("time span:", t[-1] - t[0], "s")
print("raw mean:", np.mean(y))
print("raw std:", np.std(y))

# -------------------------
# INVERT
# -------------------------
y = -y

# -------------------------
# SMOOTH
# -------------------------
window = min(11, len(y) // 2 * 2 - 1)
if window < 5:
    window = 5

y_smooth = savgol_filter(y, window, 3)

# -------------------------
# REMOVE OFFSET + DRIFT
# -------------------------
y0 = y_smooth - np.mean(y_smooth)

trend = np.polyfit(t, y0, 1)
y0 = y0 - (trend[0] * t + trend[1])

print("\nDRIFT CHECK")
print("slope:", trend[0])

# -------------------------
# FFT (omega)
# -------------------------
fft = np.fft.fft(y0)
freqs = np.fft.fftfreq(len(y0), dt)

mask = freqs > 0
dominant_freq = freqs[mask][np.argmax(np.abs(fft[mask]))]
omega = 2 * np.pi * dominant_freq

print("\nFREQUENCY")
print("f (Hz):", dominant_freq)
print("omega:", omega)

# -------------------------
# VELOCITY
# -------------------------
v = np.gradient(y0, dt)

# -------------------------
# PEAKS
# -------------------------
peaks, _ = find_peaks(y0, distance=10)
peaks_t = t[peaks]
peaks_y = np.abs(y0[peaks])

valid = peaks_y > 0.1 * np.max(peaks_y)
peaks_t = peaks_t[valid]
peaks_y = peaks_y[valid]

# -------------------------
# DAMPING
# -------------------------
gamma = -np.polyfit(peaks_t, np.log(peaks_y + 1e-12), 1)[0]

zeta = gamma / np.sqrt(omega**2 + gamma**2)

print("\nDAMPING")
print("gamma:", gamma)
print("zeta:", zeta)

# -------------------------
# SPRING CONSTANT (ASSUME MASS)
# -------------------------
m = 0.2
k = m * omega**2

print("\nSPRING CONSTANT")
print("k:", k)

# -------------------------
# ENERGY
# -------------------------
E = 0.5 * v**2 + 0.5 * omega**2 * y0**2

# -------------------------
# MODEL
# -------------------------
y_model = np.exp(-gamma * t) * np.cos(omega * t)

rmse = np.sqrt(np.mean((y0 - y_model)**2))
nrmse = rmse / (np.max(y0) - np.min(y0))

print("\nFIT QUALITY")
print("RMSE:", rmse)
print("NRMSE:", nrmse)

# -------------------------
# STABILITY (HALVES)
# -------------------------
mid = len(y0)//2

def estimate(x, tseg):
    fft = np.fft.fft(x)
    freqs = np.fft.fftfreq(len(x), dt)
    mask = freqs > 0
    w = 2*np.pi*freqs[mask][np.argmax(np.abs(fft[mask]))]

    peaks, _ = find_peaks(x, distance=10)
    pt = tseg[peaks]
    py = np.abs(x[peaks])

    g = -np.polyfit(pt, np.log(py + 1e-12), 1)[0]

    return w, g

w1, g1 = estimate(y0[:mid], t[:mid])
w2, g2 = estimate(y0[mid:], t[mid:])

print("\nSTABILITY")
print("omega drift %:", abs(w1-w2)/w2*100)
print("gamma drift %:", abs(g1-g2)/g2*100)

# -------------------------
# PLOTS
# -------------------------
# -------------------------
# PLOT 1: FIT
# -------------------------
plt.figure()
plt.plot(t, y0, label="measured signal", alpha=0.8)
plt.plot(t, y_model, label="model fit", linestyle="--")
plt.legend()
plt.title("Spring Motion: Data vs Model")
plt.xlabel("Time (s)")
plt.ylabel("Position (a.u.)")
plt.tight_layout()
plt.show()

# -------------------------
# PLOT 2: ENERGY
# -------------------------
plt.figure()
plt.plot(t, E, color="purple")
plt.title("Mechanical Energy Decay")
plt.xlabel("Time (s)")
plt.ylabel("Energy (a.u.)")
plt.tight_layout()
plt.show()

# -------------------------
# PLOT 3: PHASE SPACE
# -------------------------
plt.figure()
plt.plot(y0, v, color="black", alpha=0.8)
plt.title("Phase Space (Position vs Velocity)")
plt.xlabel("Position")
plt.ylabel("Velocity")
plt.tight_layout()
plt.show()

# -------------------------
# PLOT 4: DAMPING ENVELOPE
# -------------------------
plt.figure()
plt.scatter(peaks_t, peaks_y, s=20, label="peaks")
env = np.exp(np.polyval(np.polyfit(peaks_t, np.log(peaks_y + 1e-12), 1), peaks_t))
plt.plot(peaks_t, env, linewidth=2, label="exponential fit")

plt.title("Amplitude Decay (Damping Envelope)")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, find_peaks

plt.style.use("seaborn-v0_8-darkgrid")

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "lines.linewidth": 2,
})

# =========================================================
# PART 1 — VIDEO + TRACKING (NO CSV)
# =========================================================
video_path = "actualspring.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise RuntimeError("Video failed to open.")

ret, frame = cap.read()
if not ret:
    raise RuntimeError("Cannot read first frame.")

def create_tracker():
    if hasattr(cv2, "legacy") and hasattr(cv2.legacy, "TrackerCSRT_create"):
        return cv2.legacy.TrackerCSRT_create()
    elif hasattr(cv2, "TrackerCSRT_create"):
        return cv2.TrackerCSRT_create()
    else:
        return cv2.TrackerKCF_create()

bbox = cv2.selectROI("Select Object", frame, False)
cv2.destroyAllWindows()

tracker = create_tracker()
tracker.init(frame, bbox)

centers = []
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30

smooth_x, smooth_y = None, None
vx, vy = 0, 0

alpha = 0.35
beta = 0.75

while True:
    ret, frame = cap.read()
    if not ret:
        break

    success, bbox = tracker.update(frame)

    if success:
        x, y, w, h = map(int, bbox)
        roi = frame[y:y+h, x:x+w]

        if roi.size == 0:
            continue

        gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (5,5), 0)
        _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_OTSU)

        M = cv2.moments(mask)

        if M["m00"] != 0:
            dx = M["m10"] / M["m00"]
            dy = M["m01"] / M["m00"]
        else:
            dx, dy = w/2, h/2

        cx = x + dx
        cy = y + dy

        if smooth_x is None:
            smooth_x, smooth_y = cx, cy
        else:
            vx = beta * vx + (1-beta) * (cx - smooth_x)
            vy = beta * vy + (1-beta) * (cy - smooth_y)

            smooth_x += vx
            smooth_y += vy

            smooth_x = alpha * cx + (1-alpha) * smooth_x
            smooth_y = alpha * cy + (1-alpha) * smooth_y

        centers.append((smooth_x, smooth_y))

    cv2.imshow("Tracking", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

centers = np.array(centers)

print("\nDATA CHECK")
print("samples:", len(centers))
print("time:", len(centers)/fps)

# =========================================================
# PART 2 — PHYSICS PIPELINE
# =========================================================

t = np.arange(len(centers)) / fps
y = -centers[:, 1]   # invert image axis

# smoothing
window = min(11, len(y)//2*2 - 1)
window = max(window, 5)
y_smooth = savgol_filter(y, window, 3)

# drift removal
y0 = y_smooth - np.mean(y_smooth)
trend = np.polyfit(t, y0, 1)
y0 = y0 - (trend[0]*t + trend[1])

# velocity
v = np.gradient(y0, 1/fps)

# FFT omega
fft = np.fft.fft(y0)
freqs = np.fft.fftfreq(len(y0), d=1/fps)
mask = freqs > 0
omega = 2*np.pi*freqs[mask][np.argmax(np.abs(fft[mask]))]

# peaks
peaks, _ = find_peaks(y0, distance=10)
peaks_t = t[peaks]
peaks_y = np.abs(y0[peaks])

valid = peaks_y > 0.1*np.max(peaks_y)
peaks_t = peaks_t[valid]
peaks_y = peaks_y[valid]

gamma = -np.polyfit(peaks_t, np.log(peaks_y + 1e-12), 1)[0]

# spring constant (assumed mass)
m = 0.2
k = m * omega**2

# energy
E = 0.5*v**2 + 0.5*omega**2*y0**2

# model
y_model = np.exp(-gamma*t) * np.cos(omega*t)

# error
rmse = np.sqrt(np.mean((y0 - y_model)**2))
nrmse = rmse / (np.max(y0)-np.min(y0))

# stability
mid = len(y0)//2

def estimate(x, tseg):
    fft = np.fft.fft(x)
    f = np.fft.fftfreq(len(x), 1/fps)
    m = f > 0
    w = 2*np.pi*f[m][np.argmax(np.abs(fft[m]))]

    p, _ = find_peaks(x, distance=10)
    pt = tseg[p]
    py = np.abs(x[p])

    g = -np.polyfit(pt, np.log(py+1e-12), 1)[0]
    return w, g

w1, g1 = estimate(y0[:mid], t[:mid])
w2, g2 = estimate(y0[mid:], t[mid:])

print("\nRESULTS")
print("omega:", omega)
print("gamma:", gamma)
print("k:", k)

print("\nSTABILITY")
print("omega drift %:", abs(w1-w2)/w2*100)
print("gamma drift %:", abs(g1-g2)/g2*100)

print("\nFIT")
print("RMSE:", rmse)
print("NRMSE:", nrmse)

# =========================================================
# PART 3 — BEAUTIFUL PLOTS
# =========================================================

plt.figure()
plt.plot(t, y0, label="data")
plt.plot(t, y_model, "--", label="model")
plt.title("Spring Motion Fit")
plt.legend()
plt.show()

plt.figure()
plt.plot(t, E, color="purple")
plt.title("Energy Decay")
plt.show()

plt.figure()
plt.plot(y0, v, color="black")
plt.title("Phase Space")
plt.xlabel("x")
plt.ylabel("v")
plt.show()

plt.figure()
plt.scatter(peaks_t, peaks_y)
env = np.exp(np.polyval(np.polyfit(peaks_t, np.log(peaks_y+1e-12), 1), peaks_t))
plt.plot(peaks_t, env)
plt.title("Damping Envelope")
plt.show()